# 📊 Data Quality Observability & Intelligent Remediation
**Autonomous AI-Agent Platform for Enterprise Data Health**

▶️ Run the cell below to launch the platform. 

**Zero-Touch Deployment:**
- No IP address required
- No Auth Token required
- No Port number required

---
### 🔑 Optional: Enable Email Alerts
Add credentials via **Colab Secrets** (🔑 icon in the left sidebar) before running:
- `SENDGRID_API_KEY` 
- `DQ_ALERT_RECIPIENTS` 

In [ ]:
# @title 🚀 Launch Platform (One-Click)
import os, subprocess, threading, time, sys
from google.colab import userdata

print("🔄 Initializing Zero-Touch Setup...")

# 1. Clone repository
REPO_URL = "https://github.com/Teja-Jan/Data-Quality-Observability-Intelligent-Remediation.git"
REPO_DIR = "Data-Quality-Observability-Intelligent-Remediation"
BRANCH   = "Data-Quality-Observability-and-Intelligent-Remediation"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository...")
    !git clone -b $BRANCH $REPO_URL
else:
    print("✅ Repo already present.")
    %cd $REPO_DIR
    !git pull
    %cd ..

%cd $REPO_DIR

# 2. Install dependencies
print("📦 Installing dependencies...")
!pip install -q -r requirements.txt

# 3. Set up environment
def get_secret(key):
    try: return userdata.get(key)
    except: return ""

SG_KEY = get_secret('SENDGRID_API_KEY')
SG_TO  = get_secret('DQ_ALERT_RECIPIENTS')

with open(".env", "w") as f:
    f.write(f"EMAIL_PROVIDER={'sendgrid' if SG_KEY else 'smtp'}\n")
    f.write(f"SENDGRID_API_KEY={SG_KEY}\n")
    f.write(f"DQ_ALERT_RECIPIENTS={SG_TO}\n")
    f.write("APP_ENV=colab\n")
    f.write("LOG_LEVEL=INFO\n")
    f.write("DB_PATH=src/db/dq_metadata.db\n")

# 4. Install Cloudflare Tunnel (Zero-Touch Public URL)
print("🌐 Setting up secure tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 5. Start Ollama in background
print("🤖 Initializing AI engine...")
def setup_ollama():
    os.system("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1")
    os.system("ollama serve > /dev/null 2>&1 &")
    time.sleep(5)
    os.system("ollama pull llama3 > /dev/null 2>&1")

threading.Thread(target=setup_ollama, daemon=True).start()

# 6. Launch Streamlit
print("🚀 Launching application...")
PORT = 8501
os.system(f"fuser -k {PORT}/tcp > /dev/null 2>&1")
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.getcwd()}/src"
get_ipython().system_raw(f'streamlit run src/app.py --server.port {PORT} &')

# 7. Start Tunnel and Extract URL
print("⏳ Generating public access link...")
os.system(f"cloudflared tunnel --url http://localhost:{PORT} > tunnel.log 2>&1 &")
time.sleep(10)

tunnel_url = ""
with open("tunnel.log", "r") as f:
    log = f.read()
    for line in log.split("\n"):
        if "trycloudflare.com" in line:
            tunnel_url = "https://" + line.split("https://")[1].split(" ")[0].strip()
            break

print("\n" + "═"*60)
print("  ✅ DATA QUALITY OBSERVABILITY PLATFORM READY")
print("═"*60)
if tunnel_url:
    print(f"  🌐 Public URL:  {tunnel_url}")
else:
    # Fallback to Colab Proxy if Cloudflare fails
    from google.colab.output import eval_js
    proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"  🌐 Access Link: {proxy_url}")
print("═"*60)
print("  ℹ️  No IP, Token, or Port required. Click the link to start.")
print("═"*60 + "\n")